# 02 — Backbones pesados: ViT-B/14, e por que não ViT-L

É o notebook que a VRAM maior habilita: testa **capacidade**, o único lever que a 2060
de 8 GB não conseguia comprar.

## A justificativa, e o contra-argumento que muda o desenho

| fonte | regime | S | B | L | g |
|---|---|---|---|---|---|
| DINOv2 Tab. 11 — profundidade NYUd, RMSE ↓, **congelado** | regressão densa | 0,356 | 0,317 | 0,293 | 0,279 |
| LetsMap Tab. S.1 — BEV mIoU ↑, **fine-tune com 1% dos rótulos** | poucos rótulos | 25,55 | **28,96** | 28,40 | 28,16 |

Congelado o ganho é monotônico até ViT-g. **Sob fine-tune com poucos rótulos ele satura
em ViT-B e L/g pioram** — caso publicado de "maior é pior", e é exatamente o nosso
regime: 30.704 imagens, backbone inteiramente descongelado.

Daí o desenho **S → B**, com ViT-L atrás de um portão. Se B não bater S, L quase
certamente não bate, e você economiza metade da noite.

### Três armadilhas que quebram em silêncio

1. **ViT-L tem 24 blocos, não 12.** `unfreeze_last_n: 12` descongelaria *metade* do
   modelo. Para L use 24.
2. **Não existe layer-wise LR decay aqui.** A receita do MAE usa `layer_decay` 0,65
   (B) e 0,75 (L). O substituto que temos é a razão `backbone_lr/lr` = 1e-5/3e-4 =
   0,033, dentro da faixa 0,02–0,1 da literatura. Mantida.
3. **`drop_path` deveria subir com o tamanho** (0,1 → 0,2 no MAE). Não temos essa chave;
   `model.dropout` age só no trunk. É limitação real, anotada e não contornada.


## 1. Runtime e GPU

**Antes de rodar:** `Runtime > Change runtime type > A100 GPU`, com *High-RAM* **desligado**
— o pico medido é 482 MiB no ViT-S e ~4 GB no ViT-B, e a variante de 80 GB custa +39% de
unidades por memória que não usamos.

Custo: A100-40GB ≈ 5,4 unidades/h, então 24 h ≈ 130 CU ≈ **26% da cota mensal do Pro+**
(500 CU). Confira a taxa real em *View resources*, no menu superior direito.


In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

O pacote exige CPython >= 3.14, que o Colab normalmente não traz — por isso o `uv`
provisiona um interpretador próprio. **A instalação do torch CUDA é obrigatória e
verificada:** o extra `allsky` fixa uma wheel de CPU, e um torch de CPU aqui invalida
a sessão inteira.

Esta é a única célula que não pode vir do `_colab_runner`: é ela que clona o repo onde
o runner mora.


In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "main"
WORKDIR = "/content/micrometeorology"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "cu130",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

## 3. Dados e artefatos

`stage_bundle` copia o bundle para o SSD local, desempacota e roda `validate-dataset`.
Os três passos importam: treinar de `/content/drive` é FUSE e a leitura fria domina a
época, e um bundle truncado treinaria em silêncio sem a validação.

`ARTIFACTS` no Drive é o que sobrevive à sessão — o timeout por inatividade do Colab só
conta **quando a execução termina**, e todo run para por early stopping na época ~20.


In [ ]:
import os

import _colab_runner as runner
from google.colab import drive

BUNDLE = "/content/drive/MyDrive/labmim/allsky-mm/bundle.tar.gz"
DATA = "/content/allsky-mm"
ARTIFACTS = "/content/drive/MyDrive/labmim/runs/allsky-mm"

drive.mount("/content/drive")
os.makedirs(ARTIFACTS, exist_ok=True)
ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)

## 4. Hardware e ajustes que dependem dele

`bf16` existe em toda GPU do Colab menos a T4 (Turing). O código local roda `fp16`
porque a 2060 não tem alternativa; aqui a escolha é automática.

O probe roda no interpretador do venv, que é onde o torch com CUDA está instalado.


In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
AMP_DTYPE = HW["amp_dtype"]
WORKERS = min(8, HW["cpus"])
print(HW)
print(f"amp={AMP_DTYPE}  workers={WORKERS}")

## 5. As células de capacidade

O ViT-S aqui **não** é redundante com o notebook 01: roda na mesma sessão, mesma GPU e
mesmo staging que o ViT-B, então o delta S→B fica pareado. Comparar contra outra sessão
introduziria justamente a variável que queremos isolar.


In [ ]:
from pathlib import Path

CFG = Path(WORKDIR) / "configs/allsky/experiments/colab"
OUT = Path("/content/out")
rows = []

WHY_B = (
    "S->B e o maior passo em toda tabela do DINOv2, e o unico que sobrevive ao caso "
    "LetsMap de fine-tune com poucos rotulos"
)
ARMS = [
    ("dinov2_vits14", 12, 64, "controle pareado: a mesma referencia, nesta GPU"),
    ("dinov2_vitb14", 12, 48, WHY_B),
]


def capacity_arm(backbone, blocks, batch, note, seed):
    """Run one capacity arm, archive it, and append its metrics row."""
    tag = backbone.replace("dinov2_", "")
    name = f"cap_{tag}_s{seed}"
    config = runner.write_config(
        CFG / f"{name}.yaml",
        extends=["../_base.yaml", "../../models/image_only.yaml"],
        name=name,
        output_dir=str(OUT / name),
        seed=seed,
        data_root=ROOT,
        model={
            "backbone": backbone,
            "backbone_frozen": False,
            "unfreeze_last_n": blocks,
            "image_size": 224,
        },
        train={
            "backbone_lr": 1e-5,
            "epochs": 40,
            "batch_size": batch,
            "num_workers": WORKERS,
            "amp": {"enabled": True, "dtype": AMP_DTYPE},
        },
        targets=runner.DHI_ONLY_TARGETS,
        note=note,
    )
    row = runner.run_experiment(config)
    row["backbone"] = backbone
    print(name, row.get("status"), row.get("rmse"), row.get("wall_seconds"))
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    rows.append(row)
    return row


for backbone, blocks, batch, note in ARMS:
    for seed in (42, 43):
        capacity_arm(backbone, blocks, batch, note, seed)

runner.summarise(rows)

## 6. O portão do ViT-L

Só siga se ViT-B bater ViT-S por mais que o ruído entre sementes. Caso contrário a
literatura já disse o que vai acontecer, e a noite rende mais no notebook 03.


In [ ]:
import numpy as np


def mean_rmse(backbone):
    """Mean and seed spread of the test RMSE for one backbone."""
    values = [r["rmse"] for r in rows if r.get("status") == "ok" and r.get("backbone") == backbone]
    spread = float(np.std(values, ddof=1)) if len(values) > 1 else float("nan")
    return float(np.mean(values)), spread


s_mean, s_sd = mean_rmse("dinov2_vits14")
b_mean, b_sd = mean_rmse("dinov2_vitb14")
delta = b_mean - s_mean
noise = max(s_sd, b_sd)
print(f"ViT-S {s_mean:.2f} +- {s_sd:.2f}")
print(f"ViT-B {b_mean:.2f} +- {b_sd:.2f}")
print(f"delta {delta:+.2f} W/m2  (ruido ~{noise:.2f})")
RUN_L = bool(delta < -2 * noise)
print("rodar ViT-L?", "SIM" if RUN_L else "NAO: B nao superou S de forma mensuravel")

In [ ]:
if RUN_L:
    capacity_arm(
        "dinov2_vitl14", 24, 24, "aposta: 24 blocos, e a literatura preve piora neste regime", 42
    )
    runner.summarise(rows)
else:
    print("pulado pelo portao acima")

## 7. Resolução — o outro lever que a VRAM habilita

Os checkpoints do DINOv2 são **nativos de 518 px**: a grade posicional é 37×37, porque o
pré-treino termina com uma fase de adaptação em alta resolução. Rodar a 224 px (grade
16×16) usa a interpolação *para baixo* dessa grade — subir para 448 (32×32) interpola
**em direção** ao que os pesos viram.

A interpolação é automática no `torch.hub` (`interpolate_pos_encoding` roda a cada
forward), então é só config. A regra é ser **múltiplo de 14**: 448 sim, 512 não.

Ressalva: a Fig. 6 do DINOv2 mostra que um modelo *sem* adaptação de alta resolução
piora acima de ~336 px. Os checkpoints liberados tiveram essa fase, então esperamos
ganho — mas é A/B, não certeza.


In [ ]:
BEST = "dinov2_vitb14" if delta < 0 else "dinov2_vits14"
BLOCKS = 24 if BEST == "dinov2_vitl14" else 12
print("resolucao testada sobre:", BEST)

for px, batch in ((322, 24), (448, 12)):
    tag = BEST.replace("dinov2_", "")
    name = f"res{px}_{tag}_s42"
    config = runner.write_config(
        CFG / f"{name}.yaml",
        extends=["../_base.yaml", "../../models/image_only.yaml"],
        name=name,
        output_dir=str(OUT / name),
        seed=42,
        data_root=ROOT,
        model={
            "backbone": BEST,
            "backbone_frozen": False,
            "unfreeze_last_n": BLOCKS,
            "image_size": px,
        },
        train={
            "backbone_lr": 1e-5,
            "epochs": 40,
            "batch_size": batch,
            "num_workers": WORKERS,
            "amp": {"enabled": True, "dtype": AMP_DTYPE},
        },
        targets=runner.DHI_ONLY_TARGETS,
        note=f"{px}px = grade {px // 14}x{px // 14}, em direcao a nativa 37x37",
    )
    row = runner.run_experiment(config)
    row["backbone"] = BEST
    row["image_size"] = px
    print(name, row.get("status"), row.get("rmse"), row.get("wall_seconds"))
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    rows.append(row)

runner.summarise(rows)

In [ ]:
frame = runner.summarise(rows)
frame.to_csv(f"{ARTIFACTS}/capacidade.csv", index=False)
print(frame.to_string())
print("gravado em", ARTIFACTS)